# Evaluate Trained classifiers Models
This is the last step in the pipeline


In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)


## Load labeled data

In [ ]:
LABELED_CSV_DIR = Path("../data/labeled_logs")

csv_files = sorted(LABELED_CSV_DIR.glob("*.csv"))

dfs = [pd.read_csv(file) for file in csv_files]
df = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(csv_files)} labeled CSV file(s)")
print(f"Total rows before cleaning: {len(df)}")
df.head()


## Apply the same preprocessing/features as 
`train_classifiers.ipynb`

In [ ]:
MODEL_DIR = Path("../models")
ENCODER_PATH = MODEL_DIR / "label_encoder.pkl"

if not ENCODER_PATH.exists():
    raise FileNotFoundError(
        f"Missing label encoder: {ENCODER_PATH}. Run train_classifiers.ipynb first."
    )

le = joblib.load(ENCODER_PATH)

# Match train_classifiers.ipynb cleaning steps
df = df[df["dist"].notna()].copy()
df = df[df["light_state"] != "unknown"].copy()

# Convert is_oncoming to numeric exactly like training
if "is_oncoming" not in df.columns:
    raise KeyError("Missing required column: is_oncoming")

df["is_oncoming"] = df["is_oncoming"].astype(int)

# Use the saved encoder from training instead of fitting a new one
current_light_states = set(df["light_state"].unique())

df["light_enc"] = le.transform(df["light_state"])

features = ["class", "dist", "delta_dist", "light_enc", "is_oncoming"]
target = "is_safe"

X = df[features]
y = df[target].astype(int)

# Same split as train_classifiers.ipynb
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print(f"Rows after cleaning: {len(df)}")
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print("Class distribution:")
print(y.value_counts())

X_test.head()

## Load saved models with `joblib`
Load the trained classifier models

In [ ]:
model_paths = {
    "Logistic Regression": MODEL_DIR / "logistic_regression.pkl",
    "Decision Tree": MODEL_DIR / "decision_tree.pkl",
    "Random Forest": MODEL_DIR / "random_forest.pkl",
}

models = {name: joblib.load(path) for name, path in model_paths.items()}

print("Loaded models:")
for name, path in model_paths.items():
    print(f"- {name}: {path}")


## Run predictions + print metrics and repots

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    y_pred = ((model.predict_proba(X_test)[:, 1] >= 0.7).astype(int)
              if name == "Random Forest" else model.predict(X_test))
    predictions[name] = y_pred

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
    }

    if hasattr(model, "predict_proba") and len(set(y_test)) == 2:
        y_score = model.predict_proba(X_test)[:, 1]
        row["ROC-AUC"] = roc_auc_score(y_test, y_score)
    else:
        row["ROC-AUC"] = None

    results.append(row)

    print(f"{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")
    print(classification_report(y_test, y_pred, zero_division=0))

results_df = pd.DataFrame(results).set_index("Model")
results_df

## Confusion matrices

In [ ]:
for name, model in models.items():
    ConfusionMatrixDisplay.from_predictions(y_test, predictions[name])
    plt.title(f"{name} - Confusion Matrix")
    plt.show()

## ROC curves

In [ ]:
if len(set(y_test)) != 2:
    print("ROC curves require exactly 2 classes in y_test.")
else:
    ax = None

    for name, model in models.items():
        if hasattr(model, "predict_proba"):
            display = RocCurveDisplay.from_estimator(
                model,
                X_test,
                y_test,
                name=name,
                ax=ax,
            )
            ax = display.ax_
        else:
            print(f"Skipping {name}: model does not support predict_proba().")

    if ax is not None:
        ax.set_title("ROC Curves")
        plt.show()